# Probe Perangkat Keras — Skripsi ASR Whisper

**Tujuan:** menetapkan presisi (`compute_type`) yang dipakai seragam untuk semua varian, mengukur RTF tiap varian, dan mencari batas atas varian yang masih bisa dilatih LoRA di laptop ini.

**Tidak butuh anotasi.** Yang diukur hanya pemakaian VRAM dan kecepatan. Benar atau salahnya transkrip tidak memengaruhi keduanya. Transkrip acuan baru diperlukan saat menghitung WER, CER, dan TTER.

**Jalankan ini sebelum sapuan baseline**, karena hasilnya mengunci presisi yang dipakai seluruh eksperimen.

---

### Sebelum mulai

1. **Alihkan tampilan layar ke iGPU Radeon 780M** lewat Windows Settings → Display → Graphics, supaya 6 GB RTX 3050 utuh untuk komputasi. Cek hasilnya di sel Bagian 1 — VRAM idle idealnya di bawah 200 MB.
2. **Tutup Chrome, Discord, dan aplikasi lain yang memakai GPU.**
3. **Siapkan ruang disk.** Seluruh varian yang diunduh notebook ini bisa memakan 15–25 GB. Penyimpananmu tinggal sekitar 67 GB, jadi masih cukup, tapi jangan dijalankan saat hampir penuh.
4. **Siapkan satu berkas audio kuliah utuh** (bukan potongan 30 detik), lalu isikan namanya di sel konfigurasi.

### Catatan kejujuran

Notebook ini belum pernah dijalankan di mesinmu. Setiap tes dibungkus penanganan galat dan dijalankan di proses terpisah, jadi kegagalan satu konfigurasi tidak menjatuhkan yang lain — pesan galatnya ikut tercatat di tabel hasil. Kalau ada yang error, kirim saja tabelnya.

## 0. Dependensi

In [10]:
# Jalankan sekali. Kalau sudah terpasang, lewati saja sel ini.
# CUDA 12.x untuk RTX 3050 Laptop (Ampere).
%pip install -q peft accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Probe lingkungan

Catat keluaran sel ini apa adanya — semuanya masuk Bab 3 sebagai spesifikasi lingkungan penelitian.

In [1]:
import subprocess, sys, platform, json, os

def nvsmi(query):
    try:
        out = subprocess.check_output(
            ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL)
        return [x.strip() for x in out.decode(errors="replace").strip().splitlines()]
    except Exception as e:
        return [f"GAGAL: {e}"]

ENV = {
    "os": platform.platform(),
    "python": sys.version.split()[0],
    "gpu_name": nvsmi("name")[0],
    "vram_total_mb": nvsmi("memory.total")[0],
    "vram_idle_mb": nvsmi("memory.used")[0],
    "driver": nvsmi("driver_version")[0],
}
try:
    import torch
    ENV["torch"] = torch.__version__
    ENV["cuda_available"] = torch.cuda.is_available()
    ENV["cuda_version"] = torch.version.cuda
except Exception as e:
    ENV["torch"] = f"GAGAL: {e}"

for k, v in ENV.items():
    print(f"{k:18s}: {v}")

idle = ENV["vram_idle_mb"]
if str(idle).isdigit():
    idle = int(idle)
    if idle > 400:
        print(f"\n[PERINGATAN] VRAM idle {idle} MB terhitung tinggi.")
        print("Kemungkinan tampilan layar masih berjalan di RTX 3050, atau ada aplikasi lain memakai GPU.")
        print("Alihkan ke iGPU dan tutup aplikasi lain, lalu jalankan ulang sel ini.")
    else:
        print(f"\n[OK] VRAM idle {idle} MB. Ruang komputasi hampir penuh tersedia.")

os                : Windows-11-10.0.26200-SP0
python            : 3.12.4
gpu_name          : NVIDIA GeForce RTX 3050 6GB Laptop GPU
vram_total_mb     : 6144
vram_idle_mb      : 0
driver            : 577.02
torch             : 2.4.1+cu121
cuda_available    : True
cuda_version      : 12.1

[OK] VRAM idle 0 MB. Ruang komputasi hampir penuh tersedia.


## 2. Konfigurasi

In [3]:
from pathlib import Path

# --- WAJIB DIISI ---
AUDIO = "../data/raw/presentasi/presentasi3-audio.wav"      # satu video kuliah utuh, sudah diekstrak audionya
BEAM_SIZE = 5                    # samakan dengan nilai yang benar-benar dipakai nanti
LANGUAGE = "id"
REPEATS = 3                      # satu jalan pemanasan dibuang, lalu 3 jalan diukur, diambil nilai tengah

# --- Tangga presisi, diuji dari yang paling berat ---
PRECISION_LADDER = ["float16", "int8_float16", "int8"]

# --- Varian baseline, urut dari kecil ke besar ---
VARIANTS = ["tiny", "base", "small", "medium", "large-v3", "large-v3-turbo"]

# --- Korpus (39 video setelah duplikat ditetapkan ke APS saja) ---
CORPUS_HOURS = 12.05

OUTDIR = Path("probe_hasil"); OUTDIR.mkdir(exist_ok=True)

assert Path(AUDIO).exists(), f"Berkas audio tidak ditemukan: {AUDIO}"
print(f"Audio    : {AUDIO}")
print(f"Beam size: {BEAM_SIZE}")
print(f"Ulangan  : {REPEATS} (+1 pemanasan)")
print(f"Korpus   : {CORPUS_HOURS} jam")

Audio    : ../data/raw/presentasi/presentasi3-audio.wav
Beam size: 5
Ulangan  : 3 (+1 pemanasan)
Korpus   : 12.05 jam


## 3. Worker inferensi

Tiap konfigurasi dijalankan di **proses terpisah**. Ini penting: CTranslate2 tidak memakai alokator PyTorch, sehingga `torch.cuda.empty_cache()` tidak membebaskan memorinya. Kalau semua dijalankan dalam satu kernel, sisa memori model sebelumnya bisa menyebabkan OOM palsu pada model berikutnya.

In [4]:
WORKER_INFER = r'''
import sys, json, time, threading, subprocess

def vram_used_mb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL)
        return int(out.decode(errors="replace").strip().splitlines()[0])
    except Exception:
        return -1

def main():
    cfg = json.load(open(sys.argv[1], encoding="utf-8"))
    res = {"model": cfg["model"], "compute_type": cfg["compute_type"], "ok": False}
    idle = vram_used_mb()
    peak = [idle]
    stop = threading.Event()

    def poll():
        while not stop.is_set():
            v = vram_used_mb()
            if v > peak[0]:
                peak[0] = v
            time.sleep(0.15)

    threading.Thread(target=poll, daemon=True).start()
    try:
        from faster_whisper import WhisperModel
        t0 = time.time()
        model = WhisperModel(cfg["model"], device="cuda", compute_type=cfg["compute_type"])
        load_s = time.time() - t0
        def one_pass():
            t = time.time()
            segments, info = model.transcribe(
                cfg["audio"], beam_size=cfg["beam_size"], language=cfg["language"])
            n = 0
            for _ in segments:
                n += 1
            return time.time() - t, float(info.duration), n

        one_pass()  # pemanasan, hasilnya dibuang
        times = []
        dur, nseg = 0.0, 0
        for _ in range(int(cfg.get("repeats", 3))):
            dt, dur, nseg = one_pass()
            times.append(dt)
        times.sort()
        proc_s = times[len(times) // 2]
        res.update(ok=True,
                   load_s=round(load_s, 2),
                   proc_s=round(proc_s, 2),
                   audio_s=round(dur, 2),
                   rtf=round(proc_s / dur, 4),
                   rtf_min=round(times[0] / dur, 4),
                   rtf_max=round(times[-1] / dur, 4),
                   n_repeats=len(times),
                   n_segments=nseg)
    except Exception as e:
        res.update(ok=False, error=(type(e).__name__ + ": " + str(e))[:400])
    finally:
        stop.set()
        time.sleep(0.4)
        res["vram_idle_mb"] = idle
        res["vram_peak_mb"] = peak[0]
        res["vram_delta_mb"] = (peak[0] - idle) if idle >= 0 else -1
    sys.stdout.write("###RESULT###" + json.dumps(res))

main()
'''

Path("_worker_infer.py").write_text(WORKER_INFER, encoding="utf-8")

import tempfile, subprocess, json, sys, os

def run_infer(model, compute_type):
    cfg = {"model": model, "compute_type": compute_type, "audio": AUDIO,
           "beam_size": BEAM_SIZE, "language": LANGUAGE, "repeats": REPEATS}
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False, encoding="utf-8") as f:
        json.dump(cfg, f); cfgpath = f.name
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    try:
        p = subprocess.run([sys.executable, "_worker_infer.py", cfgpath],
                           capture_output=True, timeout=7200, env=env)
        out = p.stdout.decode(errors="replace")
        if "###RESULT###" in out:
            return json.loads(out.split("###RESULT###", 1)[1])
        return {"model": model, "compute_type": compute_type, "ok": False,
                "error": (p.stderr.decode(errors="replace") or out)[-400:]}
    except subprocess.TimeoutExpired:
        return {"model": model, "compute_type": compute_type, "ok": False, "error": "TIMEOUT >2 jam"}
    finally:
        os.unlink(cfgpath)

print("Worker siap.")

Worker siap.


## 4. Tes A — tangga presisi pada `large-v3`

`large-v3` adalah pembatasnya. Presisi pertama yang berhasil di sini dipakai untuk **seluruh** varian, termasuk `tiny`.

Unduhan pertama sekitar 3 GB dan butuh waktu. Setelah itu tersimpan di cache.

In [5]:
import pandas as pd

hasil_A = []
presisi_terkunci = None

for ct in PRECISION_LADDER:
    print(f"--- large-v3 @ {ct} ...", end=" ", flush=True)
    r = run_infer("large-v3", ct)
    hasil_A.append(r)
    if r["ok"]:
        print(f"BERHASIL | VRAM puncak {r['vram_peak_mb']} MB | RTF {r['rtf']}")
        presisi_terkunci = ct
        break
    else:
        print("GAGAL")
        print("   ", r.get("error", "")[:200])

df_A = pd.DataFrame(hasil_A)
display(df_A)

if presisi_terkunci:
    print(f"\n>>> PRESISI TERKUNCI: {presisi_terkunci}")
    print(">>> Pakai nilai ini untuk SEMUA varian di seluruh eksperimen.")
    dur = df_A[df_A["ok"] == True].iloc[-1].get("audio_s", 0)
    if dur and dur < 300:
        print(f"\n[CATATAN] Audio uji hanya {dur:.0f} detik.")
        print("  VRAM   : SAHIH. Tidak bergantung panjang audio, jadi keputusan presisi ini final.")
        print("  RTF    : SEMENTARA. Terlalu pendek untuk menangkap penurunan clock akibat panas.")
        print("           Ukur ulang dengan satu video kuliah utuh sebelum angkanya masuk Bab 3.")
else:
    print("\n>>> large-v3 tidak berjalan pada satu presisi pun.")
    print(">>> Keluarkan large-v3 dari baseline, atau jalankan di CPU (jauh lebih lambat).")
    print(">>> Lanjutkan Tes B dengan PRECISION_LADDER[-1] dan varian sampai medium saja.")

--- large-v3 @ float16 ... BERHASIL | VRAM puncak 4097 MB | RTF 0.1668


,model,compute_type,ok,load_s,proc_s,audio_s,rtf,rtf_min,rtf_max,n_repeats,n_segments,vram_idle_mb,vram_peak_mb,vram_delta_mb
0,large-v3,float16,True,893.36,38.91,233.22,0.1668,0.1667,0.1675,3,116,0,4097,4097



>>> PRESISI TERKUNCI: float16
>>> Pakai nilai ini untuk SEMUA varian di seluruh eksperimen.

[CATATAN] Audio uji hanya 233 detik.
  VRAM   : SAHIH. Tidak bergantung panjang audio, jadi keputusan presisi ini final.
  RTF    : SEMENTARA. Terlalu pendek untuk menangkap penurunan clock akibat panas.
           Ukur ulang dengan satu video kuliah utuh sebelum angkanya masuk Bab 3.


## 5. Tes B — RTF seluruh varian pada presisi terkunci

Ini yang menghasilkan tabel spesifikasi untuk Bab 3 sekaligus perkiraan jadwal sapuan baseline.

In [6]:
CT = presisi_terkunci or PRECISION_LADDER[-1]
print(f"Menjalankan semua varian pada compute_type = {CT}\n")

hasil_B = []
for m in VARIANTS:
    print(f"--- {m} ...", end=" ", flush=True)
    r = run_infer(m, CT)
    hasil_B.append(r)
    if r["ok"]:
        sebar = f" (min {r.get('rtf_min','?')} / maks {r.get('rtf_max','?')})"
        print(f"OK | VRAM {r['vram_peak_mb']} MB | RTF {r['rtf']}{sebar}")
    else:
        print("GAGAL:", r.get("error", "")[:150])

df_B = pd.DataFrame(hasil_B)
df_B.to_csv(OUTDIR / "tesB_rtf_varian.csv", index=False)
display(df_B)

Menjalankan semua varian pada compute_type = float16

--- tiny ... OK | VRAM 289 MB | RTF 0.0446 (min 0.0397 / maks 0.0574)
--- base ... OK | VRAM 449 MB | RTF 0.0474 (min 0.036 / maks 0.049)
--- small ... OK | VRAM 865 MB | RTF 0.0401 (min 0.0389 / maks 0.0414)
--- medium ... OK | VRAM 2689 MB | RTF 0.1565 (min 0.1534 / maks 0.1676)
--- large-v3 ... OK | VRAM 4097 MB | RTF 0.1706 (min 0.1704 / maks 0.1718)
--- large-v3-turbo ... OK | VRAM 2017 MB | RTF 0.0603 (min 0.0596 / maks 0.0603)


,model,compute_type,ok,load_s,proc_s,audio_s,rtf,rtf_min,rtf_max,n_repeats,n_segments,vram_idle_mb,vram_peak_mb,vram_delta_mb
0,tiny,float16,True,23.10,10.39,233.22,0.0446,0.0397,0.0574,3,45,0,289,289
1,base,float16,True,103.01,11.05,233.22,0.0474,0.0360,0.0490,3,97,0,449,449
2,small,float16,True,111.68,9.35,233.22,0.0401,0.0389,0.0414,3,42,0,865,865
3,medium,float16,True,2.73,36.50,233.22,0.1565,0.1534,0.1676,3,118,0,2689,2689
4,large-v3,float16,True,5.08,39.80,233.22,0.1706,0.1704,0.1718,3,116,0,4097,4097
5,large-v3-turbo,float16,True,3.31,14.06,233.22,0.0603,0.0596,0.0603,3,210,0,2017,2017


### 5.1 Perkiraan jadwal sapuan baseline

In [7]:
ok_B = df_B[df_B["ok"] == True].copy()
if len(ok_B):
    ok_B["jam_per_sapuan"] = (ok_B["rtf"] * CORPUS_HOURS).round(2)
    print(f"Korpus {CORPUS_HOURS} jam, satu sapuan penuh per varian:\n")
    for _, r in ok_B.iterrows():
        print(f"  {r['model']:18s} RTF {r['rtf']:.3f}  ->  {r['jam_per_sapuan']:6.2f} jam")
    total = ok_B["jam_per_sapuan"].sum()
    print(f"\n  TOTAL semua varian, satu sapuan : {total:.2f} jam")
    if "rtf_max" in ok_B.columns:
        pesimis = (ok_B["rtf_max"] * CORPUS_HOURS).sum()
        print(f"  Batas pesimistis (pakai rtf_max): {pesimis:.2f} jam")
        lebar = (ok_B["rtf_max"] / ok_B["rtf"]).max()
        if lebar > 1.25:
            print(f"\n[CATATAN] Sebaran antar-jalan sampai {lebar:.2f}x. Indikasi thermal throttling,")
            print("  jadi sapuan panjang akan lebih lambat daripada angka nilai tengah di atas.")
    print(f"  Realistis dengan jeda dan ulangan: {total*1.5:.1f} - {total*2:.1f} jam")
    if total > 40:
        print("\n[PERHATIAN] Sapuan penuh makan waktu lama. Pertimbangkan menjalankannya bertahap")
        print("per mata kuliah, dan simpan hasil transkripsi mentahnya supaya tidak perlu diulang.")
else:
    print("Belum ada varian yang berhasil. Periksa galat di Tes B.")

Korpus 12.05 jam, satu sapuan penuh per varian:

  tiny               RTF 0.045  ->    0.54 jam
  base               RTF 0.047  ->    0.57 jam
  small              RTF 0.040  ->    0.48 jam
  medium             RTF 0.157  ->    1.89 jam
  large-v3           RTF 0.171  ->    2.06 jam
  large-v3-turbo     RTF 0.060  ->    0.73 jam

  TOTAL semua varian, satu sapuan : 6.27 jam
  Batas pesimistis (pakai rtf_max): 6.60 jam

[CATATAN] Sebaran antar-jalan sampai 1.29x. Indikasi thermal throttling,
  jadi sapuan panjang akan lebih lambat daripada angka nilai tengah di atas.
  Realistis dengan jeda dan ulangan: 9.4 - 12.5 jam


## 6. Tes C — batas atas pelatihan LoRA

**Juga tidak butuh anotasi.** Yang diuji adalah alokasi memori, bukan mutu belajar, jadi transkrip bohongan sudah cukup. Konfigurasinya mengikuti Liu, Yang dan Qu (2024): LoRA pada posisi *query* dan *value*, rank 32, alpha 64.

Ingat pembandingnya — mereka memakai satu A40 48 GB dan satu A100 80 GB. Kalau batas atasmu jatuh di `small` atau `medium`, itu wajar dan bukan kesalahanmu.

In [8]:
WORKER_LORA = r'''
import sys, json, time

def main():
    cfg = json.load(open(sys.argv[1], encoding="utf-8"))
    res = {"model_id": cfg["model_id"], "batch_size": cfg["batch_size"], "ok": False}
    try:
        import torch
        from transformers import WhisperForConditionalGeneration
        from peft import LoraConfig, get_peft_model

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        model = WhisperForConditionalGeneration.from_pretrained(
            cfg["model_id"], torch_dtype=torch.float16)
        model.config.forced_decoder_ids = None
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()

        lcfg = LoraConfig(r=32, lora_alpha=64,
                          target_modules=["q_proj", "v_proj"],
                          lora_dropout=0.05, bias="none")
        model = get_peft_model(model, lcfg)
        for p in model.parameters():
            if p.requires_grad:
                p.data = p.data.float()
        model.to("cuda")

        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

        mel = model.config.num_mel_bins
        B = cfg["batch_size"]
        feats = torch.randn(B, mel, 3000, dtype=torch.float16, device="cuda")
        labels = torch.randint(100, 5000, (B, 64), device="cuda")

        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
        t0 = time.time()
        for _ in range(3):
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(input_features=feats, labels=labels).loss
            loss.backward()
            opt.step(); opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        step_s = (time.time() - t0) / 3

        res.update(ok=True,
                   num_mel_bins=mel,
                   total_params_m=round(total / 1e6, 1),
                   trainable_params_m=round(trainable / 1e6, 2),
                   trainable_pct=round(100 * trainable / total, 3),
                   peak_alloc_mb=round(torch.cuda.max_memory_allocated() / 1024**2),
                   peak_reserved_mb=round(torch.cuda.max_memory_reserved() / 1024**2),
                   sec_per_step=round(step_s, 2))
    except Exception as e:
        res.update(ok=False, error=(type(e).__name__ + ": " + str(e))[:400])
    sys.stdout.write("###RESULT###" + json.dumps(res))

main()
'''

Path("_worker_lora.py").write_text(WORKER_LORA, encoding="utf-8")

def run_lora(model_id, batch_size=1):
    cfg = {"model_id": model_id, "batch_size": batch_size}
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False, encoding="utf-8") as f:
        json.dump(cfg, f); cfgpath = f.name
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    try:
        p = subprocess.run([sys.executable, "_worker_lora.py", cfgpath],
                           capture_output=True, timeout=3600, env=env)
        out = p.stdout.decode(errors="replace")
        if "###RESULT###" in out:
            return json.loads(out.split("###RESULT###", 1)[1])
        return {"model_id": model_id, "batch_size": batch_size, "ok": False,
                "error": (p.stderr.decode(errors="replace") or out)[-400:]}
    except subprocess.TimeoutExpired:
        return {"model_id": model_id, "batch_size": batch_size, "ok": False, "error": "TIMEOUT"}
    finally:
        os.unlink(cfgpath)

print("Worker LoRA siap.")

Worker LoRA siap.


In [11]:
HF_IDS = [
    "openai/whisper-tiny",
    "openai/whisper-base",
    "openai/whisper-small",
    "openai/whisper-medium",
    "openai/whisper-large-v3-turbo",
    "openai/whisper-large-v3",
]

hasil_C = []
for mid in HF_IDS:
    print(f"--- LoRA {mid} (batch 1) ...", end=" ", flush=True)
    r = run_lora(mid, batch_size=1)
    hasil_C.append(r)
    if r["ok"]:
        print(f"OK | puncak {r['peak_reserved_mb']} MB | "
              f"{r['trainable_params_m']}M dari {r['total_params_m']}M "
              f"({r['trainable_pct']}%) | {r['sec_per_step']}s/langkah")
    else:
        err = r.get("error", "")
        tag = "OOM" if "out of memory" in err.lower() else "GAGAL"
        print(f"{tag}: {err[:150]}")

df_C = pd.DataFrame(hasil_C)
df_C.to_csv(OUTDIR / "tesC_lora_memori.csv", index=False)
display(df_C)

ok_C = df_C[df_C["ok"] == True]
if len(ok_C):
    print(f"\n>>> Varian terbesar yang bisa dilatih LoRA: {ok_C.iloc[-1]['model_id']}")
    print(">>> Varian yang di-fine-tune nanti harus dipilih dari daftar yang lolos ini.")
else:
    print("\n>>> Tidak ada yang lolos. Periksa pesan galatnya sebelum menyimpulkan.")

--- LoRA openai/whisper-tiny (batch 1) ... OK | puncak 250 MB | 0.59M dari 38.4M (1.538%) | 0.31s/langkah
--- LoRA openai/whisper-base (batch 1) ... OK | puncak 412 MB | 1.18M dari 73.8M (1.599%) | 0.21s/langkah
--- LoRA openai/whisper-small (batch 1) ... OK | puncak 1132 MB | 3.54M dari 245.3M (1.443%) | 0.34s/langkah
--- LoRA openai/whisper-medium (batch 1) ... OK | puncak 3088 MB | 9.44M dari 773.3M (1.22%) | 0.92s/langkah
--- LoRA openai/whisper-large-v3-turbo (batch 1) ... OK | puncak 3858 MB | 6.55M dari 815.4M (0.804%) | 0.91s/langkah
--- LoRA openai/whisper-large-v3 (batch 1) ... OK | puncak 5970 MB | 15.73M dari 1559.2M (1.009%) | 1.85s/langkah


,model_id,batch_size,ok,num_mel_bins,total_params_m,trainable_params_m,trainable_pct,peak_alloc_mb,peak_reserved_mb,sec_per_step
0,openai/whisper-tiny,1,True,80,38.4,0.59,1.538,219,250,0.31
1,openai/whisper-base,1,True,80,73.8,1.18,1.599,381,412,0.21
2,openai/whisper-small,1,True,80,245.3,3.54,1.443,1097,1132,0.34
3,openai/whisper-medium,1,True,80,773.3,9.44,1.220,3055,3088,0.92
4,openai/whisper-large-v3-turbo,1,True,128,815.4,6.55,0.804,3513,3858,0.91
5,openai/whisper-large-v3,1,True,128,1559.2,15.73,1.009,5489,5970,1.85



>>> Varian terbesar yang bisa dilatih LoRA: openai/whisper-large-v3
>>> Varian yang di-fine-tune nanti harus dipilih dari daftar yang lolos ini.


### 6.1 Kalau varian yang kamu inginkan kena OOM

Coba berurutan, dan hentikan di langkah pertama yang berhasil:

1. Pastikan tidak ada proses lain memakai GPU (`nvidia-smi` di terminal)
2. Perpendek `labels` dari 64 ke 32 di worker — panjang label ikut memakan memori aktivasi
3. Pasang `bitsandbytes` lalu ganti ke optimizer 8-bit `AdamW8bit`
4. Turunkan rank LoRA dari 32 ke 16, tapi **catat** perubahannya karena ini menyimpang dari konfigurasi Liu, Yang dan Qu dan wajib disebut di Bab 3

Kalau semua gagal, turun satu tingkat varian. Itu keputusan yang sah dan didokumentasikan, bukan kegagalan penelitian.

## 7. Rangkuman untuk Bab 3

In [12]:
ringkasan = {
    "lingkungan": ENV,
    "presisi_terkunci": presisi_terkunci or "TIDAK ADA - large-v3 gagal",
    "beam_size": BEAM_SIZE,
    "korpus_jam": CORPUS_HOURS,
    "tes_A_tangga_presisi": hasil_A,
    "tes_B_rtf_varian": hasil_B,
    "tes_C_lora_memori": hasil_C,
}
(OUTDIR / "ringkasan_probe.json").write_text(
    json.dumps(ringkasan, indent=2, ensure_ascii=False), encoding="utf-8")

print("Tersimpan di:", OUTDIR.resolve())
print()
print("=" * 62)
print("YANG SUDAH BISA DIPUTUSKAN SEKARANG")
print("=" * 62)
print(f"  Presisi seluruh eksperimen : {presisi_terkunci or 'perlu ditinjau ulang'}")
if len(ok_B := df_B[df_B['ok'] == True]):
    print(f"  Varian yang jalan inferensi : {', '.join(ok_B['model'].tolist())}")
if len(ok_C := df_C[df_C['ok'] == True]):
    print(f"  Varian yang bisa dilatih    : {', '.join(x.split('/')[-1] for x in ok_C['model_id'])}")
print()
print("YANG MASIH MENUNGGU ANOTASI")
print("  WER, CER, TTER, dan karena itu juga pemilihan varian yang di-fine-tune.")

Tersimpan di: C:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\notebooks\probe_hasil

YANG SUDAH BISA DIPUTUSKAN SEKARANG
  Presisi seluruh eksperimen : float16
  Varian yang jalan inferensi : tiny, base, small, medium, large-v3, large-v3-turbo
  Varian yang bisa dilatih    : whisper-tiny, whisper-base, whisper-small, whisper-medium, whisper-large-v3-turbo, whisper-large-v3

YANG MASIH MENUNGGU ANOTASI
  WER, CER, TTER, dan karena itu juga pemilihan varian yang di-fine-tune.


In [13]:
run_lora("openai/whisper-large-v3", batch_size=1)

{'model_id': 'openai/whisper-large-v3',
 'batch_size': 1,
 'ok': True,
 'num_mel_bins': 128,
 'total_params_m': 1559.2,
 'trainable_params_m': 15.73,
 'trainable_pct': 1.009,
 'peak_alloc_mb': 5489,
 'peak_reserved_mb': 5970,
 'sec_per_step': 2.52}